# 5주차 · 데이터 수집 API 가이드 (심화·선택)

수업 기본 실습은 **API 키 없이** 수업 저장소의 KRX 실제 시세 CSV로 진행합니다.
이 노트북은 학생이 **다른 종목·다른 기간**을 직접 수집할 때 쓰는 방법 B~D를 따라 하는 가이드입니다.

| 방법 | 키 | 준비 시간 | 장점 | 주의 |
|---|---|---|---|---|
| B. FinanceDataReader | 없음 | 즉시 | 코드 한 줄 | 종목 시세는 네이버 차트 서버 경유 → 접속 제한·사이트 변경 시 실패 가능 |
| C. 공공데이터포털 금융위원회_주식시세정보 | 서비스키 | 회원가입·활용신청(개발계정은 보통 자동 승인) | 정부 공식 공공데이터, 기간 조회 가능 | 전 영업일까지만 제공(1일 1회 갱신), 일일 호출 한도 |
| D. KRX Open API | AUTH_KEY | 회원가입 → 인증키 신청 → 서비스별 이용신청(승인 대기) | 거래소 공식 | 1회 호출 = 하루치 전종목 → 1년이면 약 250회 호출 |
| E. 네이버 금융 화면 | 없음 | 즉시 | 눈으로 교차검증 | 자동 수집(스크래핑)은 하지 않음 |

**마지막 단계에서 내가 수집한 값을 수업 저장소 CSV와 대조**해 수집이 제대로 됐는지 확인합니다.

> 🔐 **키 관리 원칙:** API 키를 코드 셀에 직접 쓰지 마세요(노트북을 공유하면 키가 같이 공유됩니다).
> Colab 왼쪽 🔑 **보안 비밀(Secrets)** 에 `DATA_GO_KR_KEY`, `KRX_AUTH_KEY` 이름으로 저장하고 '노트북 액세스'를 켜세요.

In [ ]:
!pip -q install finance-datareader
import time
import pandas as pd
import requests

BASE = "https://raw.githubusercontent.com/itlects/2026-2_ai-data/main/Week05/data/"
CODE, START, END = "005930", "2026-09-01", "2026-10-13"     # ← 바꿔서 실습

def get_secret(name):
    # Colab 보안 비밀 → 환경변수 순서로 키를 찾는다. 없으면 None
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        import os
        return os.environ.get(name)

def compare_with_course(mine, code=CODE):
    # 내가 수집한 Date/Close를 수업 저장소 KRX 데이터와 날짜별로 대조한다(저장소에 있는 3종목만 가능).
    try:
        ref = pd.read_csv(BASE + f"krx_daily_{code}.csv", dtype={"Code": str}, parse_dates=["Date"])
    except Exception:
        print("수업 저장소에 없는 종목입니다. 네이버 금융 화면에서 몇 개 날짜를 직접 대조하세요.")
        return None
    m = mine[["Date", "Close"]].merge(ref[["Date", "Close"]], on="Date", suffixes=("_내수집", "_KRX"))
    m["차이"] = m["Close_내수집"] - m["Close_KRX"]
    print(f"공통 날짜 {len(m)}일 · 종가가 다른 날 {int((m['차이'] != 0).sum())}일")
    return m

---
## 방법 B · FinanceDataReader
- 종목코드만 넣으면 네이버 금융에서, `'KS11'`(코스피 지수)은 FinanceData의 GitHub 캐시에서 받아옵니다.
- 실패하면 오류 메시지를 기록하고 방법 C 또는 수업 저장소 CSV를 사용합니다.

In [ ]:
import FinanceDataReader as fdr

try:
    b = fdr.DataReader(CODE, START, END).reset_index()          # 열: Date, Open, High, Low, Close, Volume, Change
    print("수집 성공:", b.shape); display(b.tail(3))
    compare_with_course(b)
except Exception as e:
    print("FinanceDataReader 종목 시세 수집 실패 →", type(e).__name__, e)

try:
    k = fdr.DataReader("KS11", START, END)
    print("KOSPI 지수 수집 성공:", k.shape)
except Exception as e:
    print("KOSPI 지수 수집 실패 →", e)

---
## 방법 C · 공공데이터포털 「금융위원회_주식시세정보」

### 키 발급 (처음 1회, 약 5분)
1. https://www.data.go.kr 회원가입 → 로그인
2. 검색창에 **금융위원회_주식시세정보** 입력 → 오픈API 상세 화면(https://www.data.go.kr/data/15094808/openapi.do)에서 **활용신청**
3. 활용목적: `교육/수업 실습` 입력 → 신청 (개발계정은 보통 자동 승인. 승인 직후 몇 분~1시간 동안은 '등록되지 않은 키' 오류가 날 수 있음)
4. **마이페이지 → 데이터활용 → Open API → 활용신청 현황** 에서 **일반 인증키(Decoding)** 복사 → Colab 보안 비밀 `DATA_GO_KR_KEY`에 저장

### 요청 형식
- 주소: `https://apis.data.go.kr/1160100/service/GetStockSecuritiesInfoService/getStockPriceInfo`
- 주요 파라미터: `serviceKey`, `resultType=json`, `numOfRows`, `pageNo`, `likeSrtnCd`(단축코드), `beginBasDt`·`endBasDt`(YYYYMMDD)
- 주요 응답 필드: `basDt`(기준일), `srtnCd`(단축코드), `itmsNm`(종목명), `mkp`(시가), `hipr`(고가), `lopr`(저가), `clpr`(종가), `trqu`(거래량), `trPrc`(거래대금), `mrktTotAmt`(시가총액), `fltRt`(등락률)
- 응답은 **문자열**, **날짜 내림차순**으로 옵니다 → 워크북 1에서 배운 전처리가 그대로 필요합니다.

In [ ]:
FSC_URL = "https://apis.data.go.kr/1160100/service/GetStockSecuritiesInfoService/getStockPriceInfo"

def fetch_fsc(code, start, end, key, rows=1000):
    # 공공데이터포털 주식시세 → Date/Open/High/Low/Close/Volume 오름차순 DataFrame
    s = pd.Timestamp(start).strftime("%Y%m%d")
    e = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y%m%d")   # 끝 날짜를 확실히 포함하도록 하루 더 요청
    items, page = [], 1
    while True:
        params = {"serviceKey": key, "resultType": "json", "numOfRows": rows, "pageNo": page,
                  "likeSrtnCd": code, "beginBasDt": s, "endBasDt": e}
        r = requests.get(FSC_URL, params=params, timeout=30)
        r.raise_for_status()
        try:
            body = r.json()["response"]["body"]
        except ValueError:                       # 키 오류 등은 XML 메시지로 온다
            raise RuntimeError("JSON이 아닌 응답(키 미등록·오타 가능): " + r.text[:200])
        got = body["items"]["item"] if body["items"] else []
        items += got if isinstance(got, list) else [got]
        if page * rows >= int(body["totalCount"]):
            break
        page += 1
    raw = pd.DataFrame(items)
    if raw.empty:
        return raw
    df = pd.DataFrame({"Date": pd.to_datetime(raw["basDt"], format="%Y%m%d"),
                       "Code": raw["srtnCd"].astype(str).str.zfill(6), "Name": raw["itmsNm"],
                       "Open": pd.to_numeric(raw["mkp"]), "High": pd.to_numeric(raw["hipr"]),
                       "Low": pd.to_numeric(raw["lopr"]), "Close": pd.to_numeric(raw["clpr"]),
                       "Volume": pd.to_numeric(raw["trqu"])})
    df = df[(df["Code"] == code) & (df["Date"] <= pd.Timestamp(end))]
    return df.drop_duplicates("Date").sort_values("Date").reset_index(drop=True)

key = get_secret("DATA_GO_KR_KEY")
if not key:
    print("DATA_GO_KR_KEY가 없습니다. 위 '키 발급' 절차 후 Colab 보안 비밀에 저장하고 다시 실행하세요.")
else:
    c = fetch_fsc(CODE, START, END, key)
    print("수집:", c.shape); display(c.tail(3))
    compare_with_course(c)
    c.to_csv(f"fsc_{CODE}.csv", index=False, encoding="utf-8-sig")

### C-2. 지수시세 (워크북 4용 · 금융위원회_지수시세정보)
- https://www.data.go.kr/data/15094807/openapi.do 에서 **별도로 활용신청** (같은 인증키 사용)
- 파라미터: `idxNm`(예: `코스피`, `코스닥`), `beginBasDt`, `endBasDt` 등
- 서비스 주소는 아래 값을 쓰되, 오류가 나면 활용가이드(docx)의 요청 주소와 대조합니다. 응답 열 이름은 출력으로 확인합니다.

In [ ]:
IDX_URL = "https://apis.data.go.kr/1160100/service/GetMarketIndexInfoService/getStockMarketIndex"

def fetch_index(name, start, end, key, rows=1000):
    s = pd.Timestamp(start).strftime("%Y%m%d")
    e = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y%m%d")
    items, page = [], 1
    while True:
        r = requests.get(IDX_URL, params={"serviceKey": key, "resultType": "json", "numOfRows": rows,
                                          "pageNo": page, "idxNm": name, "beginBasDt": s, "endBasDt": e}, timeout=30)
        r.raise_for_status()
        body = r.json()["response"]["body"]
        got = body["items"]["item"] if body["items"] else []
        items += got if isinstance(got, list) else [got]
        if page * rows >= int(body["totalCount"]):
            break
        page += 1
    raw = pd.DataFrame(items)
    print("응답 열:", list(raw.columns))
    if raw.empty:
        return raw
    raw = raw[raw["idxNm"] == name]
    df = pd.DataFrame({"Date": pd.to_datetime(raw["basDt"], format="%Y%m%d"),
                       "Close": pd.to_numeric(raw["clpr"].astype(str).str.replace(",", ""))})
    return df[df["Date"] <= pd.Timestamp(end)].drop_duplicates("Date").sort_values("Date").reset_index(drop=True)

if key:
    kospi_api = fetch_index("코스피", START, END, key)
    ref = pd.read_csv(BASE + "kospi_daily.csv", parse_dates=["Date"])
    display(kospi_api.merge(ref[["Date", "Close"]], on="Date", suffixes=("_API", "_수업자료")).tail())

---
## 방법 D · KRX Open API (한국거래소)

### 키 발급
1. http://openapi.krx.co.kr 회원가입 → 로그인 → **마이페이지 → API 인증키 신청** (승인 후 인증키 확인)
2. **서비스 이용 → 주식 → 유가증권 일별매매정보** 에서 **이용신청** (서비스마다 따로 신청, 승인까지 시간이 걸릴 수 있음)
3. 인증키를 Colab 보안 비밀 `KRX_AUTH_KEY`에 저장

### 요청 형식
- 주소: `https://data-dbg.krx.co.kr/svc/apis/sto/stk_bydd_trd` (코스닥은 `ksq_bydd_trd`)
- 파라미터: `AUTH_KEY`, `basDd`(YYYYMMDD) → **그날 유가증권시장 전 종목**이 `OutBlock_1` 목록으로 옴
- 주요 필드: `BAS_DD`, `ISU_CD`(종목코드), `ISU_NM`, `TDD_OPNPRC`(시가), `TDD_HGPRC`, `TDD_LWPRC`, `TDD_CLSPRC`(종가), `ACC_TRDVOL`(거래량), `MKTCAP`
- 하루 1회 호출이므로 기간이 길면 호출 수가 많아집니다. 아래 예시는 **거래일 목록**을 수업 저장소 KOSPI 파일에서 가져와 그 날만 호출합니다.

In [ ]:
KRX_URL = "https://data-dbg.krx.co.kr/svc/apis/sto/stk_bydd_trd"

def to_num(x):
    return pd.to_numeric(str(x).replace(",", ""), errors="coerce")

def fetch_krx(code, dates, key):
    rows = []
    for d in dates:
        r = requests.get(KRX_URL, params={"AUTH_KEY": key, "basDd": d.strftime("%Y%m%d")}, timeout=30)
        r.raise_for_status()
        for it in r.json().get("OutBlock_1", []):
            if str(it.get("ISU_CD", "")).endswith(code) or it.get("ISU_SRT_CD") == code:
                rows.append({"Date": pd.to_datetime(it["BAS_DD"], format="%Y%m%d"), "Code": code, "Name": it["ISU_NM"],
                             "Open": to_num(it["TDD_OPNPRC"]), "High": to_num(it["TDD_HGPRC"]),
                             "Low": to_num(it["TDD_LWPRC"]), "Close": to_num(it["TDD_CLSPRC"]),
                             "Volume": to_num(it["ACC_TRDVOL"])})
        time.sleep(0.2)                                   # 서버 배려: 연속 호출 간격
    return pd.DataFrame(rows).sort_values("Date").reset_index(drop=True)

krx_key = get_secret("KRX_AUTH_KEY")
if not krx_key:
    print("KRX_AUTH_KEY가 없습니다. 위 '키 발급' 절차를 마친 뒤 다시 실행하세요.")
else:
    cal = pd.read_csv(BASE + "kospi_daily.csv", parse_dates=["Date"])["Date"]
    days = cal[(cal >= START) & (cal <= END)]
    print(f"{len(days)}거래일 호출 시작…")
    k = fetch_krx(CODE, days, krx_key)
    print("수집:", k.shape); display(k.tail(3))
    compare_with_course(k)

---
## 방법 E · 네이버 금융으로 교차검증 (화면 확인)
- 일별 시세: `https://finance.naver.com/item/sise_day.naver?code=005930`
- 위에서 수집한 마지막 3일 종가를 화면과 비교해 워크북 1 체크리스트에 적습니다.
- 네이버 금융은 이용약관상 무단 자동 수집이 제한될 수 있으므로 **requests/BeautifulSoup 스크래핑은 하지 않습니다.**

### 정리 질문
1. 방법 B·C·D 중 실제로 성공한 방법과 실패한 방법, 실패 메시지는?
2. 수업 저장소 값과 다른 날이 있었다면 원인은 무엇일까? (기준일 차이, 수정주가 여부, 장중 데이터 등)
3. 공공데이터가 '전 영업일까지'만 제공된다는 점은 분석 결과 해석에 어떤 영향을 주는가?